# Incident Response Runbook: Impact - Data Destruction

**Tactic:** Impact
**Technique:** Data Destruction (T1485)
**Severity:** CRITICAL

## Overview

This runbook provides step-by-step incident response procedures for Data Destruction activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add the project root to the Python path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()

# Query Splunk for data destruction indicators
print(f"\n[QUERY] Searching Splunk for data destruction indicators...")
splunk_query = '''
index=* (sourcetype=WinEventLog:Security EventCode=4660 OR EventCode=4663 OR EventCode=4670)
OR (sourcetype=fs_notification "delete" OR "remove" OR "unlink" OR "rm" OR "del")
OR (sourcetype=WinEventLog:Security EventCode=4659 AccessMask=0x10000)
OR (sourcetype=linux_secure "rm" OR "shred" OR "wipe" OR "dd" "zero")
OR (sourcetype=WinEventLog:Security EventCode=5145 ShareName!="*IPC$" AccessMask=0x10000)
| eval file_operation=coalesce(Operation, AccessMask, action)
| eval file_path=coalesce(ObjectName, file_path, FileName, Path)
| eval destructive_ops=if(match(file_operation, "DELETE|0x10000|remove|unlink|rm|shred|wipe"), "true", "false")
| where destructive_ops="true"
| stats count by host, user, file_path, file_operation, _time
| where count > 10
'''

try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-24h")
    print(f"   Found {len(splunk_results)} potential data destruction events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

# Extract affected systems and destruction activities
affected_systems = []
destruction_activities = []
splunk_indicators = []
unique_users = set()
total_files_deleted = 0

for event in splunk_results:
    system_info = {
        'hostname': event.get('host', 'unknown'),
        'user': event.get('user', 'unknown'),
        'file_path': event.get('file_path', ''),
        'operation': event.get('file_operation', ''),
        'event_count': event.get('count', 0),
        'last_seen': event.get('_time', detection_time)
    }
    affected_systems.append(system_info)
    unique_users.add(event.get('user', 'unknown'))
    total_files_deleted += event.get('count', 0)

    # Extract indicators
    if event.get('file_path'):
        splunk_indicators.append({
            'type': 'file_deletion',
            'value': event.get('file_path'),
            'context': f"File deleted by {event.get('user', 'unknown')} on {event.get('host', 'unknown')}"
        })

# Query CrowdStrike for endpoint detection
print(f"\n[QUERY] Checking CrowdStrike for data destruction detections...")
try:
    cs_detections = crowdstrike.get_detections(
        filter="technique:'Data Destruction'",
        start_time="-24h"
    )
    cs_affected_hosts = []
    for detection in cs_detections:
        host_info = {
            'device_id': detection.get('device_id', ''),
            'hostname': detection.get('hostname', ''),
            'detection_id': detection.get('detection_id', ''),
            'technique': detection.get('technique', ''),
            'severity': detection.get('severity', 0)
        }
        cs_affected_hosts.append(host_info)

        # Merge with Splunk data
        existing_system = next((s for s in affected_systems if s['hostname'] == host_info['hostname']), None)
        if existing_system:
            existing_system.update(host_info)
        else:
            affected_systems.append(host_info)

    print(f"   Found {len(cs_affected_hosts)} CrowdStrike detections")
except Exception as e:
    print(f"   CrowdStrike query failed: {e}")
    cs_affected_hosts = []

# Enrich with threat intelligence from MISP
print(f"\n[ENRICHMENT] Checking MISP for related threat intelligence...")
misp_results = []
try:
    for indicator in splunk_indicators[:10]:  # Check first 10 indicators
        misp_search = misp.search_iocs(indicator['value'])
        if misp_search:
            misp_results.extend(misp_search)
            indicator['threat_intel'] = misp_search[0] if misp_search else None
            print(f"   Found threat intelligence for destructive activity")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    incident_data = {
        'title': f'Data Destruction Incident - {len(affected_systems)} systems, {total_files_deleted} files deleted',
        'description': f'Detected data destruction activities on {len(affected_systems)} systems affecting {total_files_deleted} files',
        'severity': 'CRITICAL',
        'tactic': 'Impact',
        'technique': 'Data Destruction (T1485)',
        'indicators': splunk_indicators,
        'affected_systems': affected_systems,
        'threat_intelligence': misp_results
    }

    incident_id = iris.create_case(incident_data)
    if incident_id:
        print(f"   Created IRIS case: {incident_id}")
    else:
        print(f"   Failed to create IRIS case")
        incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n Detection complete:")
print(f"  - Affected systems: {len(affected_systems)}")
print(f"  - Unique users: {len(unique_users)}")
print(f"  - Total files deleted: {total_files_deleted}")
print(f"  - Suspicious indicators: {len(splunk_indicators)}")
print(f"  - Threat intelligence hits: {len(misp_results)}")
print(f"  - Incident ID: {incident_id}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
containment_actions = []
isolated_hosts = []
disabled_accounts = []
blocked_ips = []

# 1. IMMEDIATE: Stop destructive processes
print(f"\n[CONTAINMENT] Stopping destructive processes immediately...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Get running processes that might be destructive
            processes = crowdstrike.get_processes(system['device_id'])
            destructive_procs = []

            for proc in processes:
                cmd_line = proc.get('command_line', '').lower()
                if any(destructive_cmd in cmd_line for destructive_cmd in ['rm ', 'del ', 'shred', 'wipe', 'format', 'cipher /w']):
                    destructive_procs.append(proc)

            for proc in destructive_procs:
                kill_result = crowdstrike.kill_process(system['device_id'], proc['process_id'])
                if kill_result:
                    containment_actions.append({
                        'action': 'process_kill',
                        'target': f"{system['hostname']}:{proc['process_name']}",
                        'method': 'CrowdStrike',
                        'status': 'success',
                        'timestamp': containment_time
                    })
                    print(f"   Killed destructive process: {proc['process_name']} on {system['hostname']}")
except Exception as e:
    print(f"   Error stopping destructive processes: {e}")

# 2. Isolate affected systems
print(f"\n[CONTAINMENT] Isolating affected systems...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Use CrowdStrike to isolate host
            isolation_result = crowdstrike.isolate_host(system['device_id'])
            if isolation_result:
                isolated_hosts.append(system['hostname'])
                containment_actions.append({
                    'action': 'host_isolation',
                    'target': system['hostname'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Isolated host: {system['hostname']}")
            else:
                print(f"   Failed to isolate host: {system['hostname']}")
        else:
            # Use Shuffle for network isolation
            network_isolation = shuffle.isolate_system(system['hostname'])
            if network_isolation:
                isolated_hosts.append(system['hostname'])
                containment_actions.append({
                    'action': 'network_isolation',
                    'target': system['hostname'],
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Network isolated: {system['hostname']}")
except Exception as e:
    print(f"   Error during host isolation: {e}")

# 3. Disable suspicious accounts
print(f"\n[CONTAINMENT] Disabling suspicious accounts...")
try:
    for user in unique_users:
        if user and user != 'unknown':
            # Disable account via Shuffle
            disable_result = shuffle.disable_account(user)
            if disable_result:
                disabled_accounts.append(user)
                containment_actions.append({
                    'action': 'account_disable',
                    'target': user,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Disabled account: {user}")
except Exception as e:
    print(f"   Error disabling accounts: {e}")

# 4. Enable enhanced monitoring and backups
print(f"\n[CONTAINMENT] Enabling enhanced monitoring and backup protection...")
try:
    monitoring_rules = [
        {
            'name': 'Critical Data Destruction Monitoring',
            'query': 'index=* (EventCode=4660 OR EventCode=4663 OR EventCode=4659) | eval risk_score = case(match(ObjectName, ".*\\\\(Users|Documents|Desktop|ProgramData)\\\\.*"), 10, match(ObjectName, ".*\\.(docx|xlsx|pdf|db|bak)$"), 9, 1==1, 7) | where risk_score >= 8',
            'alert_threshold': 1,
            'time_window': '1m'
        }
    ]

    enhanced_monitoring = splunk.enable_enhanced_monitoring(monitoring_rules)
    if enhanced_monitoring:
        containment_actions.append({
            'action': 'enhanced_monitoring',
            'target': 'Splunk',
            'method': 'Splunk',
            'status': 'success',
            'timestamp': containment_time
        })
        print(f"   Enabled critical data destruction monitoring in Splunk")

    # Enable backup protection
    backup_protection = shuffle.enable_backup_protection(affected_systems)
    if backup_protection:
        containment_actions.append({
            'action': 'backup_protection',
            'target': 'All affected systems',
            'method': 'Shuffle',
            'status': 'success',
            'timestamp': containment_time
        })
        print(f"   Enabled backup protection for affected systems")

except Exception as e:
    print(f"   Error enabling monitoring: {e}")

print(f"\n Containment complete:")
print(f"  - Hosts isolated: {len(isolated_hosts)}")
print(f"  - Accounts disabled: {len(disabled_accounts)}")
print(f"  - Destructive processes stopped: ")
print(f"  - Enhanced monitoring: enabled")
print(f"  - Backup protection: enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
eradication_actions = []
terminated_processes = []
deleted_tools = []
reset_credentials = []

# 1. Remove destructive malware and tools
print(f"\n[ERADICATION] Removing destructive malware and tools...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Scan for and remove common destructive tools
            tools_to_remove = [
                'sdelete.exe', 'cipher.exe', 'ccleaner.exe', 'eraser.exe',
                'wiper.bat', 'destroy.ps1', 'shred.exe', 'wipe.exe'
            ]

            for tool in tools_to_remove:
                removal_result = crowdstrike.remove_file(system['device_id'], tool)
                if removal_result:
                    deleted_tools.append(f"{system['hostname']}:{tool}")
                    eradication_actions.append({
                        'action': 'tool_removal',
                        'target': f"{system['hostname']}:{tool}",
                        'method': 'CrowdStrike',
                        'status': 'success',
                        'timestamp': eradication_time
                    })
                    print(f"   Removed destructive tool: {tool} from {system['hostname']}")
except Exception as e:
    print(f"   Error removing tools: {e}")

# 2. Reset compromised credentials
print(f"\n[ERADICATION] Resetting compromised credentials...")
try:
    for user in unique_users:
        if user and user != 'unknown':
            # Reset password via Shuffle
            reset_result = shuffle.reset_password(user)
            if reset_result:
                reset_credentials.append(user)
                eradication_actions.append({
                    'action': 'credential_reset',
                    'target': user,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Reset credentials for: {user}")
except Exception as e:
    print(f"   Error resetting credentials: {e}")

# 3. Clean up destructive scripts and scheduled tasks
print(f"\n[ERADICATION] Cleaning up destructive scripts and scheduled tasks...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Remove scheduled destructive tasks
            task_cleanup = crowdstrike.cleanup_scheduled_tasks(system['device_id'], 'destructive')
            if task_cleanup:
                eradication_actions.append({
                    'action': 'task_cleanup',
                    'target': system['hostname'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Cleaned scheduled tasks on: {system['hostname']}")

            # Remove destructive scripts
            script_cleanup = crowdstrike.remove_malicious_scripts(system['device_id'])
            if script_cleanup:
                eradication_actions.append({
                    'action': 'script_cleanup',
                    'target': system['hostname'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Removed malicious scripts from: {system['hostname']}")
except Exception as e:
    print(f"   Error cleaning up scripts and tasks: {e}")

# 4. Verify eradication
print(f"\n[ERADICATION] Verifying eradication...")
try:
    verification_results = []
    for system in affected_systems:
        if system.get('device_id'):
            # Run verification scan for destructive indicators
            scan_result = crowdstrike.scan_system(system['device_id'], 'destructive_indicators')
            verification_results.append({
                'system': system['hostname'],
                'scan_result': scan_result,
                'clean': scan_result.get('destructive_tools_found', 0) == 0
            })

    clean_systems = [r for r in verification_results if r['clean']]
    eradication_actions.append({
        'action': 'eradication_verification',
        'target': f"{len(clean_systems)}/{len(verification_results)} systems clean",
        'method': 'CrowdStrike',
        'status': 'success' if len(clean_systems) == len(verification_results) else 'partial',
        'timestamp': eradication_time
    })

    print(f"   Verification complete: {len(clean_systems)}/{len(verification_results)} systems clean")

except Exception as e:
    print(f"   Error during verification: {e}")

# 5. Assess data loss impact
print(f"\n[ERADICATION] Assessing data loss impact...")
try:
    data_loss_assessment = {
        'total_files_deleted': total_files_deleted,
        'affected_file_types': list(set([ind['value'].split('.')[-1] if '.' in ind['value'] else 'unknown' for ind in splunk_indicators[:20]])),
        'critical_data_affected': any('Users' in ind['value'] or 'Documents' in ind['value'] for ind in splunk_indicators),
        'backup_availability': shuffle.check_backup_availability(affected_systems)
    }

    eradication_actions.append({
        'action': 'data_loss_assessment',
        'target': f"Data loss assessment completed",
        'method': 'Analysis',
        'status': 'success',
        'timestamp': eradication_time
    })

    print(f"   Data loss assessment complete:")
    print(f"    - Total files deleted: {data_loss_assessment['total_files_deleted']}")
    print(f"    - Critical data affected: {data_loss_assessment['critical_data_affected']}")
    print(f"    - Backup available: {data_loss_assessment['backup_availability']}")

except Exception as e:
    print(f"   Error assessing data loss: {e}")

print(f"\n Eradication complete:")
print(f"  - Tools deleted: {len(deleted_tools)}")
print(f"  - Credentials reset: {len(reset_credentials)}")
print(f"  - Scripts/tasks cleaned: ")
print(f"  - Systems verified clean: {len([r for r in verification_results if r.get('clean', False)])}")
print(f"  - Data loss assessed: ")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
recovery_actions = []
reenabled_hosts = []
restored_accounts = []
data_restored = []

# 1. Restore data from backups
print(f"\n[RECOVERY] Restoring data from backups...")
try:
    for system in affected_systems:
        # Attempt data restoration from backups
        restore_result = shuffle.restore_from_backup(system['hostname'])
        if restore_result:
            data_restored.append(system['hostname'])
            recovery_actions.append({
                'action': 'data_restoration',
                'target': system['hostname'],
                'method': 'Shuffle',
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Restored data for: {system['hostname']}")
        else:
            print(f"   No backup available for: {system['hostname']}")
except Exception as e:
    print(f"   Error during data restoration: {e}")

# 2. Re-enable isolated systems
print(f"\n[RECOVERY] Re-enabling isolated systems...")
try:
    for host in isolated_hosts:
        system = next((s for s in affected_systems if s['hostname'] == host), None)
        if system and system.get('device_id'):
            # Use CrowdStrike to re-enable host
            reenable_result = crowdstrike.reenable_host(system['device_id'])
            if reenable_result:
                reenabled_hosts.append(host)
                recovery_actions.append({
                    'action': 'host_reenable',
                    'target': host,
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Re-enabled host: {host}")
            else:
                print(f"   Failed to re-enable host: {host}")
        else:
            # Use Shuffle for network re-enablement
            network_restore = shuffle.restore_system(host)
            if network_restore:
                reenabled_hosts.append(host)
                recovery_actions.append({
                    'action': 'network_restore',
                    'target': host,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Restored network access: {host}")
except Exception as e:
    print(f"   Error re-enabling systems: {e}")

# 3. Restore disabled accounts
print(f"\n[RECOVERY] Restoring disabled accounts...")
try:
    for user in disabled_accounts:
        # Restore account access via Shuffle
        restore_result = shuffle.restore_account(user)
        if restore_result:
            restored_accounts.append(user)
            recovery_actions.append({
                'action': 'account_restore',
                'target': user,
                'method': 'Shuffle',
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Restored account: {user}")
except Exception as e:
    print(f"   Error restoring accounts: {e}")

# 4. Validate system functionality
print(f"\n[RECOVERY] Validating system functionality...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Validate system health after restoration
            health_check = crowdstrike.validate_system_health(system['device_id'])
            if health_check and health_check.get('status') == 'healthy':
                recovery_actions.append({
                    'action': 'system_validation',
                    'target': system['hostname'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   System health validated: {system['hostname']}")
            else:
                print(f"   System health issues detected: {system['hostname']}")
except Exception as e:
    print(f"   Error validating system health: {e}")

# 5. Restore monitoring and alerting
print(f"\n[RECOVERY] Restoring monitoring and alerting...")
try:
    # Restore normal Splunk monitoring rules
    normal_monitoring = splunk.restore_normal_monitoring()
    if normal_monitoring:
        recovery_actions.append({
            'action': 'monitoring_restore',
            'target': 'Splunk',
            'method': 'Splunk',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal monitoring in Splunk")

    # Restore CrowdStrike normal operations
    cs_normal_ops = crowdstrike.restore_normal_operations()
    if cs_normal_ops:
        recovery_actions.append({
            'action': 'cs_normal_ops',
            'target': 'CrowdStrike',
            'method': 'CrowdStrike',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal CrowdStrike operations")

except Exception as e:
    print(f"   Error restoring monitoring: {e}")

# 6. Verify data integrity
print(f"\n[RECOVERY] Verifying data integrity...")
try:
    integrity_checks = []
    for system in affected_systems:
        # Check data integrity after restoration
        integrity_result = shuffle.verify_data_integrity(system['hostname'])
        integrity_checks.append({
            'system': system['hostname'],
            'integrity_ok': integrity_result.get('integrity_verified', False),
            'corrupted_files': integrity_result.get('corrupted_files', 0)
        })

    systems_with_integrity = [c for c in integrity_checks if c['integrity_ok']]
    recovery_actions.append({
        'action': 'data_integrity_check',
        'target': f"{len(systems_with_integrity)}/{len(integrity_checks)} systems with verified integrity",
        'method': 'Shuffle',
        'status': 'success' if len(systems_with_integrity) == len(integrity_checks) else 'partial',
        'timestamp': recovery_time
    })

    print(f"   Data integrity check complete: {len(systems_with_integrity)}/{len(integrity_checks)} systems OK")

except Exception as e:
    print(f"   Error verifying data integrity: {e}")

print(f"\n Recovery complete:")
print(f"  - Data restored: {len(data_restored)} systems")
print(f"  - Hosts re-enabled: {len(reenabled_hosts)}")
print(f"  - Accounts restored: {len(restored_accounts)}")
print(f"  - Systems validated: ")
print(f"  - Data integrity verified: {len([c for c in integrity_checks if c.get('integrity_ok', False)])} systems")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []
closure_time = datetime.now().isoformat()

# 1. Generate comprehensive incident report
print(f"\n[POST-INCIDENT] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'title': f'Data Destruction Incident Report - {len(affected_systems)} systems, {total_files_deleted} files affected',
        'detection_time': detection_time,
        'closure_time': closure_time,
        'severity': 'CRITICAL',
        'tactic': 'Impact',
        'technique': 'Data Destruction (T1485)',
        'summary': {
            'affected_users': len(unique_users),
            'affected_systems': len(affected_systems),
            'total_files_deleted': total_files_deleted,
            'data_restored': len(data_restored),
            'hosts_isolated': len(isolated_hosts),
            'accounts_disabled': len(disabled_accounts),
            'tools_deleted': len(deleted_tools),
            'credentials_reset': len(reset_credentials)
        },
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': eradication_time,
            'recovery': recovery_time,
            'closure': closure_time
        },
        'actions_taken': {
            'detection': splunk_indicators[:10],  # Top 10 indicators
            'containment': containment_actions,
            'eradication': eradication_actions,
            'recovery': recovery_actions
        },
        'threat_intelligence': [i.get('threat_intel') for i in splunk_indicators if i.get('threat_intel')],
        'data_loss_assessment': {
            'total_files_deleted': total_files_deleted,
            'critical_data_affected': any('Users' in ind['value'] or 'Documents' in ind['value'] for ind in splunk_indicators),
            'backup_success_rate': len(data_restored) / len(affected_systems) if affected_systems else 0,
            'data_integrity_verified': len([c for c in integrity_checks if c.get('integrity_ok', False)]) / len(integrity_checks) if integrity_checks else 0
        },
        'recommendations': [
            'Implement comprehensive backup and recovery procedures',
            'Deploy data loss prevention (DLP) systems',
            'Regular security awareness training on data destruction threats',
            'Enhance monitoring for file deletion and destructive activities',
            'Implement least privilege access controls',
            'Regular backup integrity testing and validation'
        ]
    }

    # Save report to file
    report_filename = f"data_destruction_report_{incident_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(report_filename, 'w') as f:
        json.dump(incident_report, f, indent=2, default=str)

    post_incident_actions.append({
        'action': 'report_generation',
        'target': report_filename,
        'status': 'success',
        'timestamp': closure_time
    })
    print(f"   Generated incident report: {report_filename}")

except Exception as e:
    print(f"   Error generating report: {e}")

# 2. Document lessons learned
print(f"\n[POST-INCIDENT] Documenting lessons learned...")
try:
    lessons_learned = {
        'incident_id': incident_id,
        'what_went_well': [
            'Rapid detection of data destruction activities',
            'Immediate containment prevented further data loss',
            'Comprehensive eradication of destructive tools',
            'Successful data restoration from backups'
        ],
        'what_could_improve': [
            'Earlier detection of destructive behavior patterns',
            'Enhanced backup frequency and testing',
            'Better user training on recognizing destructive activities',
            'Automated data integrity monitoring'
        ],
        'key_findings': [
            f'Data destruction affected {len(affected_systems)} systems with {total_files_deleted} files deleted',
            f'Backup restoration successful for {len(data_restored)}/{len(affected_systems)} systems',
            'Threat intelligence enriched detection for destructive malware',
            'Immediate process termination prevented complete data loss'
        ],
        'preventive_measures': [
            'Implement automated backup solutions with integrity checking',
            'Deploy advanced endpoint protection with data destruction prevention',
            'Regular security awareness training focused on data protection',
            'Enhanced monitoring and alerting for destructive activities',
            'Implement data classification and protection policies',
            'Regular backup testing and disaster recovery drills'
        ]
    }

    # Save lessons learned
    lessons_filename = f"data_destruction_lessons_{incident_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(lessons_filename, 'w') as f:
        json.dump(lessons_learned, f, indent=2, default=str)

    post_incident_actions.append({
        'action': 'lessons_learned',
        'target': lessons_filename,
        'status': 'success',
        'timestamp': closure_time
    })
    print(f"   Documented lessons learned: {lessons_filename}")

except Exception as e:
    print(f"   Error documenting lessons learned: {e}")

# 3. Implement preventive measures
print(f"\n[POST-INCIDENT] Implementing preventive measures...")
try:
    preventive_measures = []

    # Update Splunk correlation rules for data destruction
    updated_rules = splunk.update_correlation_rules([
        {
            'name': 'Enhanced Data Destruction Detection',
            'search': 'index=* (EventCode=4660 OR EventCode=4663 OR EventCode=4659) | eval risk_score = case(match(ObjectName, ".*\\\\(Users|Documents|Desktop)\\\\.*"), 10, match(ObjectName, ".*\\.(docx|xlsx|pdf|db)$"), 9, match(ObjectName, ".*\\.(bak|zip|rar)$"), 8, 1==1, 6) | where risk_score >= 8',
            'alert_threshold': 1,
            'time_window': '2m'
        }
    ])
    if updated_rules:
        preventive_measures.append('Updated Splunk data destruction rules')
        print(f"   Updated Splunk correlation rules for data destruction detection")

    # Enhance CrowdStrike prevention policies
    cs_policies = crowdstrike.update_prevention_policies({
        'data_destruction_prevention': 'enabled',
        'file_deletion_monitoring': 'strict',
        'destructive_tool_detection': 'enabled',
        'backup_protection': 'enabled'
    })
    if cs_policies:
        preventive_measures.append('Enhanced CrowdStrike data protection policies')
        print(f"   Enhanced CrowdStrike prevention policies")

    # Schedule data destruction security training
    training_scheduled = shuffle.schedule_security_training(
        users=list(unique_users),
        topic='Data Protection and Destruction Prevention',
        incident_reference=incident_id
    )
    if training_scheduled:
        preventive_measures.append('Scheduled data protection training')
        print(f"   Scheduled security awareness training on data destruction prevention")

    # Implement enhanced backup procedures
    backup_enhancement = shuffle.enhance_backup_procedures()
    if backup_enhancement:
        preventive_measures.append('Enhanced backup procedures')
        print(f"   Enhanced backup and recovery procedures")

except Exception as e:
    print(f"   Error implementing preventive measures: {e}")

# 4. Share threat intelligence
print(f"\n[POST-INCIDENT] Sharing threat intelligence...")
try:
    shared_intel = []
    for indicator in splunk_indicators[:20]:  # Share top destructive indicators
        if indicator.get('threat_intel'):
            # Share with MISP
            result = misp.share_indicator(indicator, incident_id)
            if result:
                shared_intel.append(f"MISP: {indicator.get('type', 'unknown')}")
                print(f"   Shared indicator with MISP: {indicator.get('type', 'unknown')}")

    if shared_intel:
        post_incident_actions.append({
            'action': 'threat_intel_sharing',
            'target': shared_intel,
            'status': 'success',
            'timestamp': closure_time
        })

except Exception as e:
    print(f"   Error sharing threat intelligence: {e}")

# 5. Close incident case
print(f"\n[POST-INCIDENT] Closing incident case...")
try:
    if incident_id:
        closure_data = {
            'status': 'closed',
            'closure_time': closure_time,
            'closure_reason': 'Data destruction incident successfully contained, data restored where possible, and preventive measures implemented',
            'post_incident_actions': post_incident_actions,
            'final_assessment': {
                'threat_contained': len(isolated_hosts) > 0,
                'threat_eradicated': len(deleted_tools) > 0,
                'data_recovered': len(data_restored) > 0,
                'preventive_measures': len(preventive_measures) > 0,
                'data_loss_mitigated': len([c for c in integrity_checks if c.get('integrity_ok', False)]) > 0
            }
        }

        result = iris.close_case(incident_id, closure_data)
        if result:
            post_incident_actions.append({
                'action': 'case_closure',
                'target': incident_id,
                'status': 'success',
                'timestamp': closure_time
            })
            print(f"   Closed IRIS case: {incident_id}")
        else:
            print(f"   Failed to close IRIS case: {incident_id}")

except Exception as e:
    print(f"   Error closing incident case: {e}")

print(f"\n Post-incident activities complete:")
print(f"  - Incident report generated")
print(f"  - Lessons learned documented")
print(f"  - {len(preventive_measures)} preventive measures implemented")
print(f"  - Threat intelligence shared with {len(shared_intel)} platforms")
print(f"  - Incident case closed: {incident_id}")

print(f"\n Data Destruction Incident Response Complete")
print(f"   Total duration: {(datetime.fromisoformat(closure_time) - datetime.fromisoformat(detection_time)).total_seconds() / 3600:.1f} hours")
print(f"   Data recovery rate: {len(data_restored)}/{len(affected_systems)} systems")
print(f"   Response effectiveness: {'HIGH' if len(data_restored) > 0 and len(isolated_hosts) > 0 else 'MEDIUM'}")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
